In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (keep full dataframes in variables for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamp is integer and create timezone-aware datetime column (UTC -> convert to Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_logs_df):
    df['timestamp'] = df['timestamp'].astype(int)
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Normalize KPI column name for grouping
metric_df = metric_df.rename(columns={'kpi_name': 'kpi_name'})
trace_df = trace_df.rename(columns={'trace_name': 'kpi_name'})
log_df = log_df.rename(columns={'log_name': 'kpi_name'})

# Ensure 'value' is numeric where applicable
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Helper function to build aggregates for a dataframe with columns ['cmdb_id','kpi_name','value','dt']
def build_aggregates(df):
    grp_val = df.groupby(['cmdb_id', 'kpi_name'])['value']
    count = grp_val.count().rename('count')
    p50 = grp_val.quantile(0.5).rename('p50')
    p95 = grp_val.quantile(0.95).rename('p95')
    p99 = grp_val.quantile(0.99).rename('p99')
    # timestamps
    grp_dt = df.groupby(['cmdb_id', 'kpi_name'])['dt']
    earliest = grp_dt.min().rename('earliest_ts')
    latest = grp_dt.max().rename('latest_ts')
    agg = pd.concat([count, earliest, latest, p50, p95, p99], axis=1).reset_index()
    # sort by count desc and keep top 50 for compactness
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

metric_aggregates = build_aggregates(metric_df).head(50)
trace_aggregates = build_aggregates(trace_df).head(50)
log_aggregates = build_aggregates(log_df).head(50)

# Error logs: count of messages per cmdb_id and earliest timestamp per cmdb_id
error_count = error_logs_df.groupby('cmdb_id').agg(
    count=('message', 'count'),
    earliest_ts=('dt', 'min')
).reset_index().sort_values('count', ascending=False).reset_index(drop=True)

error_logs_agg = error_count

# Display the compact aggregated tables (top 50 each)
metric_aggregates, trace_aggregates, log_aggregates, error_logs_agg.head(50)

```
Out[1]:
```
```python
# Summarize the previous aggregation results in plain English as a compact string.
# Reuse existing variables: metric_aggregates, trace_aggregates, log_aggregates, error_logs_agg

def fmt_row(r):
    # r is a Series from the aggregates with expected columns:
    # ['cmdb_id','kpi_name','count','earliest_ts','latest_ts','p50','p95','p99']
    cmdb = str(r.get('cmdb_id', ''))
    kpi = str(r.get('kpi_name', ''))
    cnt = int(r.get('count', 0))
    e = r.get('earliest_ts')
    l = r.get('latest_ts')
    e_s = pd.to_datetime(e).strftime('%Y-%%m-%d %H:%M:%S%z') if not pd.isna(e) else 'N/A'
    l_s = pd.to_datetime(l).strftime('%Y-%%m-%d %H:%M:%S%z') if not pd.isna(l) else 'N/A'
    p95 = r.get('p95')
    p95_s = f"{p95:.6g}" if pd.notna(p95) else 'N/A'
    return f"{cmdb} | {kpi} | count={cnt} | earliest={e_s} | latest={l_s} | p95={p95_s}"

lines = []
# Overview counts
lines.append(f"Metric aggregates: {len(metric_aggregates)} groups (top 50 shown).")
if len(metric_aggregates):
    lines.append("Top 3 metric groups by count:")
    for _, row in metric_aggregates.head(3).iterrows():
        lines.append("  - " + fmt_row(row))

lines.append("")
lines.append(f"Trace aggregates: {len(trace_aggregates)} groups (top 50 shown).")
if len(trace_aggregates):
    lines.append("Top 3 trace groups by count:")
    for _, row in trace_aggregates.head(3).iterrows():
        lines.append("  - " + fmt_row(row))

lines.append("")
lines.append(f"Log aggregates: {len(log_aggregates)} groups (top 50 shown).")
if len(log_aggregates):
    lines.append("Top 3 log groups by count:")
    for _, row in log_aggregates.head(3).iterrows():
        lines.append("  - " + fmt_row(row))

lines.append("")
# Error logs summary
if error_logs_agg is None or error_logs_agg.empty:
    lines.append("Error logs aggregation: no error log messages found (empty result).")
else:
    lines.append(f"Error logs aggregation: {len(error_logs_agg)} cmdb_id(s) with messages.")
    lines.append("Top cmdb_id(s) by error count:")
    for _, r in error_logs_agg.head(5).iterrows():
        cmdb = r['cmdb_id']
        cnt = int(r['count'])
        e = r['earliest_ts']
        e_s = pd.to_datetime(e).strftime('%Y-%%m-%d %H:%M:%S%z') if pd.notna(e) else 'N/A'
        lines.append(f"  - {cmdb} | count={cnt} | earliest={e_s}")

summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count               earliest_ts                 latest_ts           p50           p95           p99
0   apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  6.181178e+06  6.182001e+06  6.182073e+06
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  7.200000e+01  7.600000e+01  7.600000e+01
2    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00
3   apache02                            OSLinux-CPU_CPU_CPULoad     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  1.000000e-02  8.000000e-02  1.110000e-01
4   apache02                        OSLinux-CPU_CPU_CPUidleutil     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  9.947281e+01  9.966575e+01  9.969936e+01
..       ...                                                ...    ...                       ...                       ...           ...           ...           ...
45  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  5.000000e+02  5.000000e+02  5.000000e+02
46  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00
47  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00
48  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00
49  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 8 columns],    cmdb_id                         kpi_name  count               earliest_ts                 latest_ts        p50         p95         p99
0     IG01         trace.self.duration_mean     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.059299    0.128725    0.144502
1     IG01          trace.self.duration_p95     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.165025    0.539715    0.566430
2     IG01             trace.self.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  81.000000  142.500000  165.940000
3     IG01  trace.to_Tomcat01.duration_mean     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.050072    0.130506    0.141838
4     IG01   trace.to_Tomcat01.duration_p95     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.139000    0.510552    0.583709
..     ...                              ...    ...                       ...                       ...        ...         ...         ...
45    MG01  trace.to_dockerB2.duration_mean     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.026259    0.125856    0.130699
46    MG01      trace.to_dockerB1.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  10.500000   21.100000   22.000000
47    MG01   trace.to_dockerB1.duration_p95     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.198575    0.419573    0.554973
48    MG01   trace.to_dockerB2.duration_p95     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.086050    0.491840    0.519555
49    MG01             trace.self.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  66.500000  122.400000  148.720000

[50 rows x 8 columns],      cmdb_id          kpi_name  count               earliest_ts                 latest_ts     p50      p95      p99
0   Tomcat01   log.error_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.000   0.0000   0.0000
1   Tomcat02     log.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  23.500  42.4000  46.0000
2   Tomcat02   log.error_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.000   0.0000   0.0000
3   Tomcat01     log.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  19.500  39.5500  40.7100
4   Tomcat04     log.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  22.000  32.0000  39.1000
5   Tomcat04   log.error_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.000   0.0000   0.0000
6   Tomcat03     log.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  20.000  38.6500  41.4200
7   Tomcat03   log.error_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.000   0.0000   0.0000
8   apache02     log.row_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00  39.500  68.7000  80.6800
9   apache02   log.error_count     30 2021-03-06 06:00:00+08:00 2021-03-06 06:29:00+08:00   0.000   0.0000   0.0000
10   Mysql01     log.row_count      7 2021-03-06 06:11:00+08:00 2021-03-06 06:25:00+08:00   1.000   1.0000   1.0000
11   Mysql01   log.error_count      7 2021-03-06 06:11:00+08:00 2021-03-06 06:25:00+08:00   0.000   0.0000   0.0000
12  Tomcat02  log.gc_pause_max      5 2021-03-06 06:05:00+08:00 2021-03-06 06:28:00+08:00   0.070   0.0880   0.0896
13  Tomcat01  log.gc_pause_max      3 2021-03-06 06:04:00+08:00 2021-03-06 06:25:00+08:00   0.070   0.0790   0.0798
14  Tomcat04  log.gc_pause_max      3 2021-03-06 06:04:00+08:00 2021-03-06 06:26:00+08:00   0.070   0.0700   0.0700
15  Tomcat03  log.gc_pause_max      2 2021-03-06 06:03:00+08:00 2021-03-06 06:19:00+08:00   0.055   0.0595   0.0599, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reload metric_df if not present
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is int and create a fresh timezone-aware datetime column in UTC+8
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure kpi_name column exists
if 'kpi_name' not in metric_df.columns and 'kpi' in metric_df.columns:
    metric_df = metric_df.rename(columns={'kpi': 'kpi_name'})

metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure metric_aggregates exists; prefer to use the existing one produced earlier.
# If it's missing, compute global p95 thresholds from the full metric_df (per RULE 9 we should use global series).
if 'metric_aggregates' not in globals() or metric_aggregates is None or metric_aggregates.empty:
    grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
    metric_aggregates = pd.concat([
        grp.count().rename('count'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].min().rename('earliest_ts'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].max().rename('latest_ts'),
        grp.quantile(0.5).rename('p50'),
        grp.quantile(0.95).rename('p95'),
        grp.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)

# Prepare p95 lookup (use p95 from metric_aggregates)
p95_df = metric_aggregates[['cmdb_id', 'kpi_name', 'p95']].copy()
p95_df['p95'] = pd.to_numeric(p95_df['p95'], errors='coerce')

# Define incident window in UTC+8 (inclusive)
start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))

# Filter metric_df to the incident window (inclusive)
metric_window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# Join global p95 from metric_aggregates (match on cmdb_id and kpi_name)
metric_with_p95 = metric_window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# Keep only rows where we have a global p95 threshold
metric_with_p95 = metric_with_p95[metric_with_p95['p95'].notna()].copy()

# Mark anomaly if value >= p95 (per instruction, apply >= p95 for all KPIs)
metric_with_p95['is_anomaly'] = metric_with_p95['value'] >= metric_with_p95['p95']

# Store full anomalies dataframe in kernel variable for later steps
anomalies_metric_df = metric_with_p95[metric_with_p95['is_anomaly']].copy()

# Build compact summary table per group
if not anomalies_metric_df.empty:
    agg = anomalies_metric_df.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('dt','min'),
        latest_anomaly_ts = ('dt','max'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    ).reset_index()
    # severity = max_value / p95, handle p95 == 0
    def compute_sev(row):
        if pd.isna(row['p95']):
            return np.nan
        if row['p95'] == 0:
            return np.inf
        return row['max_value'] / row['p95']
    agg['severity'] = agg.apply(compute_sev, axis=1)
    anomalies_summary_df = agg.sort_values('severity', ascending=False).head(20).reset_index(drop=True)
else:
    anomalies_summary_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95','severity'])

# Final output: compact anomalies summary table (top 20 by severity) and keep anomalies_metric_df in kernel
anomalies_summary_df

```
Out[2]:
```
```python
# Summarize the anomalies_summary_df into a plain-English compact string.
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing anomalies_summary_df from kernel; if missing, provide a fallback message.
if 'anomalies_summary_df' not in globals():
    summary = "No anomalies_summary_df found in the kernel."
else:
    df = anomalies_summary_df.copy()
    total_groups = len(df)
    inf_count = int(np.isinf(df['severity']).sum()) if total_groups > 0 else 0
    finite_count = total_groups - inf_count

    lines = []
    lines.append("Anomaly summary for metrics in window 2021-03-06 06:00 to 06:30 (UTC+8):")
    lines.append(f"- Detected anomaly groups: {total_groups} (finite severity: {finite_count}, infinite severity: {inf_count})")
    if inf_count > 0:
        lines.append("  * Note: 'infinite' severity means global p95 == 0 for that KPI, so max_value / p95 is infinite.")

    # List top up to 5 groups by severity (df already sorted by severity desc)
    top_n = df.head(5)
    if top_n.empty:
        lines.append("- No anomaly rows to display.")
    else:
        lines.append("- Top anomaly groups (by severity):")
        for _, r in top_n.iterrows():
            cmdb = r['cmdb_id']
            kpi = r['kpi_name']
            cnt = int(r['anomaly_count'])
            p95 = r['p95']
            maxv = r['max_value']
            sev = r['severity']
            # format timestamps
            e_ts = pd.to_datetime(r['earliest_anomaly_ts']).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            l_ts = pd.to_datetime(r['latest_anomaly_ts']).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            if pd.isna(p95):
                p95_s = "N/A"
            elif p95 == 0:
                p95_s = "0 (causes infinite severity)"
            else:
                p95_s = f"{p95:.6g}"
            if np.isinf(sev):
                sev_s = "inf"
            else:
                sev_s = f"{sev:.3f}"
            lines.append(f"  - {cmdb} | {kpi} | anomalies={cnt} | earliest={e_ts} | latest={l_ts} | max={maxv:.6g} | p95={p95_s} | severity={sev_s}")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value         p95  severity
0    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
1    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
2    Redis01         redis-Redis_6379_Redis  (connected_slaves)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
3    Redis01             redis-Redis_6379_Redis  (evicted_keys)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
4   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
5    Redis01          redis-Redis_6379_Redis  (pubsub_channels)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
6   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
7   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
8   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
9   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
10   Redis01              redis-Redis_6379_Redis  (aof_enabled)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
11   Redis01  redis-Redis_6379_Redis  (aof_rewrite_in_progress)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
12   Redis01                  redis-Redis_6379_Redis  (loading)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
13   Redis01         redis-Redis_6379_Redis  (latest_fork_usec)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
14   Redis01    redis-Redis_6379_Redis  (aof_rewrite_scheduled)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
15   Redis01          redis-Redis_6379_Redis  (blocked_clients)             31 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00     0.0000    0.000000       inf
16  apache02                             OSLinux-CPU_CPU_CPUWio              2 2021-03-06 06:24:00+08:00 2021-03-06 06:30:00+08:00     0.6144    0.311350  1.973342
17   Redis01                             OSLinux-CPU_CPU_CPUWio              2 2021-03-06 06:13:00+08:00 2021-03-06 06:24:00+08:00     0.9267    0.488215  1.898139
18  apache02                            OSLinux-CPU_CPU_CPULoad              2 2021-03-06 06:01:00+08:00 2021-03-06 06:07:00+08:00     0.1200    0.080000  1.500000
19   Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...              2 2021-03-06 06:17:00+08:00 2021-03-06 06:23:00+08:00   504.0000  434.500000  1.159954```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is loaded
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = metric_df['timestamp'].astype(int)
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure global metric_aggregates with p95 exists (compute if missing)
if 'metric_aggregates' not in globals() or metric_aggregates is None or metric_aggregates.empty:
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_aggregates = pd.concat([
        grp.count().rename('count'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].min().rename('earliest_ts'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].max().rename('latest_ts'),
        grp.quantile(0.5).rename('p50'),
        grp.quantile(0.95).rename('p95'),
        grp.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)

# Ensure anomalies_metric_df exists; if not, recompute anomalies within the incident window
if 'anomalies_metric_df' not in globals() or anomalies_metric_df is None:
    # Incident window (inclusive) in UTC+8
    start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
    end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))
    # Ensure dt exists in metric_df
    if 'dt' not in metric_df.columns:
        metric_df['timestamp'] = metric_df['timestamp'].astype(int)
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()
    # Prepare p95 lookup
    p95_df = metric_aggregates[['cmdb_id','kpi_name','p95']].copy()
    p95_df['p95'] = pd.to_numeric(p95_df['p95'], errors='coerce')
    metric_with_p95 = metric_window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    metric_with_p95 = metric_with_p95[metric_with_p95['p95'].notna()].copy()
    metric_with_p95['is_anomaly'] = metric_with_p95['value'] >= metric_with_p95['p95']
    anomalies_metric_df = metric_with_p95[metric_with_p95['is_anomaly']].copy()

# Work on anomalies_metric_df to find consecutive runs
# Ensure dt is timezone-aware and sorted
an = anomalies_metric_df.copy()
an = an.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

# Compute time difference in seconds within each (cmdb_id, kpi_name)
an['dt_diff_sec'] = an.groupby(['cmdb_id','kpi_name'])['dt'].diff().dt.total_seconds().fillna(0)

# Start a new run when dt_diff_sec != 60
an['new_run_flag'] = (an['dt_diff_sec'] != 60).astype(int)
an['run_id'] = an.groupby(['cmdb_id','kpi_name'])['new_run_flag'].cumsum()

# Aggregate runs
group_cols = ['cmdb_id','kpi_name','run_id']
runs = an.groupby(group_cols).agg(
    run_start_ts = ('dt','min'),
    run_end_ts   = ('dt','max'),
    run_length   = ('dt','size'),
    anomaly_count= ('dt','size'),
    max_value    = ('value','max'),
    p95          = ('p95','first')
).reset_index()

# Compute severity and breach_ratio
def compute_severity(maxv, p95v):
    if pd.isna(p95v):
        return np.nan
    if p95v == 0:
        return np.inf
    return maxv / p95v

def compute_breach_ratio(maxv, p95v):
    if pd.isna(p95v) or p95v == 0:
        return np.nan
    return (maxv - p95v) / p95v

runs['severity'] = runs.apply(lambda r: compute_severity(r['max_value'], r['p95']), axis=1)
runs['breach_ratio'] = runs.apply(lambda r: compute_breach_ratio(r['max_value'], r['p95']), axis=1)

# Exclude runs of length 1 that have breach_ratio <= 0.5 (keep those with NaN breach_ratio)
mask_exclude = (runs['run_length'] == 1) & (runs['breach_ratio'].notna()) & (runs['breach_ratio'] <= 0.5)
fault_runs_df = runs[~mask_exclude].copy()

# Prepare summary table limited to top 20 by severity descending
fault_runs_summary_df = fault_runs_df.sort_values('severity', ascending=False).head(20).reset_index(drop=True)

# Keep only requested columns in order
fault_runs_summary_df = fault_runs_summary_df[[
    'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','anomaly_count',
    'max_value','p95','severity','breach_ratio'
]]

# Final output (compact top 20 runs). Also fault_runs_df stored in kernel.
fault_runs_summary_df

```
Out[3]:
```
```python
# Summarize the fault_runs_summary_df in plain English as a compact string.
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'fault_runs_summary_df' not in globals():
    summary = "No fault_runs_summary_df found in the kernel."
else:
    df = fault_runs_summary_df.copy()
    total_runs = len(df)
    inf_sev = int(np.isinf(df['severity']).sum())
    finite_sev = total_runs - inf_sev

    lines = []
    lines.append(f"Detected fault runs in incident window (top {min(20,total_runs)} shown): {total_runs} runs total.")
    lines.append(f"- Runs with infinite severity (p95 == 0): {inf_sev}. These indicate the global p95 for the KPI is zero.")
    lines.append(f"- Runs with finite severity: {finite_sev}.")

    # Top 5 runs by severity (df already sorted by severity desc)
    top_n = df.head(5)
    if top_n.empty:
        lines.append("- No runs to display.")
    else:
        lines.append("- Top runs by severity:")
        for _, r in top_n.iterrows():
            cmdb = r['cmdb_id']
            kpi = r['kpi_name']
            start = pd.to_datetime(r['run_start_ts']).tz_convert(tz).strftime('%Y-03-%d %H:%M:%S%z') if not pd.isna(r['run_start_ts']) else 'N/A'
            end = pd.to_datetime(r['run_end_ts']).tz_convert(tz).strftime('%Y-03-%d %H:%M:%S%z') if not pd.isna(r['run_end_ts']) else 'N/A'
            length = int(r['run_length'])
            maxv = r['max_value']
            p95 = r['p95']
            sev = r['severity']
            if pd.isna(p95):
                p95_s = "N/A"
            elif p95 == 0:
                p95_s = "0 (inf severity)"
            else:
                p95_s = f"{p95:.6g}"
            sev_s = "inf" if np.isinf(sev) else f"{sev:.3f}"
            lines.append(f"  - {cmdb} | {kpi} | start={start} | end={end} | len={length} | max={maxv:.6g} | p95={p95_s} | severity={sev_s}")

    lines.append("")
    lines.append("Notes:")
    lines.append("- Infinite severity means p95==0 so max/p95 is unbounded; inspect those KPIs (many are Redis flags or zero-baseline metrics).")
    lines.append("- Short isolated runs of length 1 with small breaches were filtered out per the rule (length 1 & breach_ratio <= 0.5).")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name              run_start_ts                run_end_ts  run_length  anomaly_count  max_value       p95  severity  breach_ratio
0    Redis01  redis-Redis_6379_Redis  (aof_rewrite_in_progress) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
1    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
2    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
3   Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou... 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
4   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
5   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
6    Redis01         redis-Redis_6379_Redis  (connected_slaves) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
7    Redis01          redis-Redis_6379_Redis  (blocked_clients) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
8    Redis01    redis-Redis_6379_Redis  (aof_rewrite_scheduled) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
9    Redis01              redis-Redis_6379_Redis  (aof_enabled) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
10  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES... 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
11   Redis01          redis-Redis_6379_Redis  (pubsub_channels) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
12   Redis01                  redis-Redis_6379_Redis  (loading) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
13   Redis01         redis-Redis_6379_Redis  (latest_fork_usec) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
14  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE... 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
15   Redis01             redis-Redis_6379_Redis  (evicted_keys) 2021-03-06 06:00:00+08:00 2021-03-06 06:30:00+08:00          31             31   0.000000  0.000000       inf           NaN
16  apache02                             OSLinux-CPU_CPU_CPUWio 2021-03-06 06:24:00+08:00 2021-03-06 06:24:00+08:00           1              1   0.614400  0.311350  1.973342      0.973342
17   Redis01                             OSLinux-CPU_CPU_CPUWio 2021-03-06 06:24:00+08:00 2021-03-06 06:24:00+08:00           1              1   0.926700  0.488215  1.898139      0.898139
18   Mysql02                             OSLinux-CPU_CPU_CPUWio 2021-03-06 06:18:00+08:00 2021-03-06 06:19:00+08:00           2              2   4.949508  4.818477  1.027194      0.027194
19   Redis01  redis-Redis_6379_Redis  (mem_fragmentation_ratio) 2021-03-06 06:04:00+08:00 2021-03-06 06:05:00+08:00           2              2   0.730000  0.720000  1.013889      0.013889```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load trace_df if missing
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp, dt, and numeric value
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Normalize trace name column to 'trace_name' for consistency
if 'trace_name' not in trace_df.columns and 'kpi_name' in trace_df.columns:
    trace_df = trace_df.rename(columns={'kpi_name': 'trace_name'})

# Ensure trace_aggregates (global thresholds) exists; if not, compute from full trace_df BEFORE filtering (per rule 9)
if 'trace_aggregates' not in globals() or trace_aggregates is None or trace_aggregates.empty:
    grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
    trace_aggregates = pd.concat([
        grp.count().rename('count'),
        trace_df.groupby(['cmdb_id','trace_name'])['dt'].min().rename('earliest_ts'),
        trace_df.groupby(['cmdb_id','trace_name'])['dt'].max().rename('latest_ts'),
        grp.quantile(0.5).rename('p50'),
        grp.quantile(0.95).rename('p95'),
        grp.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)

# Normalize trace_aggregates column name for trace key
if 'trace_name' not in trace_aggregates.columns and 'kpi_name' in trace_aggregates.columns:
    trace_aggregates = trace_aggregates.rename(columns={'kpi_name': 'trace_name'})

# Prepare p95 lookup
p95_df = trace_aggregates[['cmdb_id', 'trace_name', 'p95']].copy()
p95_df['p95'] = pd.to_numeric(p95_df['p95'], errors='coerce')

# Incident window in UTC+8 (inclusive)
start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))

# Filter trace_df to window
trace_window_df = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# Join global p95 (match on cmdb_id and trace_name)
trace_with_p95 = trace_window_df.merge(p95_df, on=['cmdb_id','trace_name'], how='left')

# Keep rows where p95 exists
trace_with_p95 = trace_with_p95[trace_with_p95['p95'].notna()].copy()

# Mark anomaly if value >= p95
trace_with_p95['is_anomaly'] = trace_with_p95['value'] >= trace_with_p95['p95']

# Store full anomalous trace points
anomalies_trace_df = trace_with_p95[trace_with_p95['is_anomaly']].copy()

# Build compact summary table per (cmdb_id, trace_name)
if not anomalies_trace_df.empty:
    agg = anomalies_trace_df.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('dt','min'),
        latest_anomaly_ts = ('dt','max'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    ).reset_index()
    # severity = max_value / p95, handle p95 == 0
    def sev_fn(row):
        p = row['p95']
        m = row['max_value']
        if pd.isna(p):
            return np.nan
        if p == 0:
            return np.inf
        return m / p
    agg['severity'] = agg.apply(sev_fn, axis=1)
    anomalies_trace_summary_df = agg.sort_values('severity', ascending=False).head(20).reset_index(drop=True)
else:
    anomalies_trace_summary_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95','severity'
    ])

# Final output: compact summary (top 20) and keep anomalies_trace_df in kernel for later steps
anomalies_trace_summary_df

```
Out[4]:
```
```python
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'anomalies_trace_summary_df' not in globals():
    summary = "No anomalies_trace_summary_df found in the kernel."
else:
    df = anomalies_trace_summary_df.copy()
    total = len(df)
    lines = []
    lines.append(f"Trace anomaly summary for window 2021-03-06 06:00–06:30 (UTC+8): {total} groups (top 20 shown).")
    if total == 0:
        lines.append("- No anomalous trace groups detected.")
    else:
        lines.append("- Top anomaly groups by severity:")
        for _, r in df.head(5).iterrows():
            cmdb = r['cmdb_id']
            tname = r['trace_name']
            cnt = int(r['anomaly_count'])
            e = pd.to_datetime(r['earliest_anomaly_ts']).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(r['earliest_anomaly_ts']) else 'N/A'
            l = pd.to_datetime(r['latest_anomaly_ts']).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(r['latest_anomaly_ts']) else 'N/A'
            maxv = r['max_value']
            p95 = r['p95']
            sev = r['severity']
            p95_s = "0 (inf severity)" if pd.notna(p95) and p95 == 0 else (f"{p95:.6g}" if pd.notna(p95) else "N/A")
            sev_s = "inf" if np.isinf(sev) else f"{sev:.3f}"
            lines.append(f"  - {cmdb} | {tname} | anomalies={cnt} | earliest={e} | latest={l} | max={maxv:.6g} | p95={p95_s} | severity={sev_s}")
        lines.append("")
        lines.append("Notes:")
        lines.append("- Highest severities observed for MG01 and IG02 trace links (e.g., MG01 trace.from_Tomcat04.duration_mean severity ~4.61).")
        lines.append("- Many groups have only 2 points in-window but exceed their global p95 significantly; inspect these trace paths for latency regressions.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value        p95  severity
0     MG01  trace.from_Tomcat04.duration_mean              2 2021-03-06 06:01:00+08:00 2021-03-06 06:17:00+08:00   0.416000   0.090166  4.613687
1     MG01  trace.from_Tomcat03.duration_mean              2 2021-03-06 06:17:00+08:00 2021-03-06 06:18:00+08:00   0.355571   0.125798  2.826529
2     IG02     trace.to_Tomcat04.duration_p95              2 2021-03-06 06:18:00+08:00 2021-03-06 06:20:00+08:00   1.174850   0.418250  2.808966
3     MG02   trace.from_Tomcat03.duration_p95              2 2021-03-06 06:16:00+08:00 2021-03-06 06:21:00+08:00   1.937150   0.702512  2.757460
4     MG02  trace.from_Tomcat03.duration_mean              2 2021-03-06 06:09:00+08:00 2021-03-06 06:16:00+08:00   0.409500   0.155664  2.630669
5     IG02    trace.to_Tomcat04.duration_mean              2 2021-03-06 06:01:00+08:00 2021-03-06 06:18:00+08:00   0.300125   0.132726  2.261239
6     MG01   trace.from_Tomcat03.duration_p95              2 2021-03-06 06:17:00+08:00 2021-03-06 06:18:00+08:00   1.071100   0.536350  1.997017
7     IG02     trace.to_Tomcat02.duration_p95              2 2021-03-06 06:05:00+08:00 2021-03-06 06:15:00+08:00   0.708300   0.387365  1.828508
8     IG01    trace.to_Tomcat03.duration_mean              2 2021-03-06 06:16:00+08:00 2021-03-06 06:17:00+08:00   0.375000   0.214847  1.745429
9     MG01   trace.from_Tomcat01.duration_p95              2 2021-03-06 06:16:00+08:00 2021-03-06 06:18:00+08:00   0.542250   0.310932  1.743948
10    IG01     trace.to_Tomcat03.duration_p95              2 2021-03-06 06:16:00+08:00 2021-03-06 06:17:00+08:00   1.481400   0.850387  1.742029
11    MG01  trace.from_Tomcat01.duration_mean              2 2021-03-06 06:04:00+08:00 2021-03-06 06:16:00+08:00   0.159500   0.095378  1.672287
12    MG01            trace.self.duration_p95              2 2021-03-06 06:01:00+08:00 2021-03-06 06:17:00+08:00   0.399200   0.267865  1.490303
13    MG01     trace.to_dockerB1.duration_p95              2 2021-03-06 06:17:00+08:00 2021-03-06 06:19:00+08:00   0.609000   0.419573  1.451477
14    MG01      trace.from_Tomcat03.row_count              3 2021-03-06 06:15:00+08:00 2021-03-06 06:23:00+08:00  28.000000  20.000000  1.400000
15    IG02        trace.to_Tomcat04.row_count              2 2021-03-06 06:15:00+08:00 2021-03-06 06:26:00+08:00  23.000000  17.100000  1.345029
16    MG01      trace.from_Tomcat04.row_count              2 2021-03-06 06:20:00+08:00 2021-03-06 06:21:00+08:00  23.000000  17.100000  1.345029
17    MG01   trace.from_Tomcat04.duration_p95              2 2021-03-06 06:01:00+08:00 2021-03-06 06:11:00+08:00   0.423200   0.315870  1.339792
18    MG02  trace.from_Tomcat02.duration_mean              2 2021-03-06 06:11:00+08:00 2021-03-06 06:27:00+08:00   0.169125   0.126763  1.334182
19    IG02     trace.to_Tomcat01.duration_p95              2 2021-03-06 06:17:00+08:00 2021-03-06 06:18:00+08:00   0.706500   0.537620  1.314125```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure anomalies_trace_summary_df exists; if not, recompute using existing anomalies_trace_df or by recomputing anomalies.
if 'anomalies_trace_summary_df' not in globals() or anomalies_trace_summary_df is None:
    # Recompute using anomalies_trace_df if available
    if 'anomalies_trace_df' in globals() and anomalies_trace_df is not None:
        src = anomalies_trace_df.copy()
        # Aggregate to summary per (cmdb_id, trace_name) as done previously
        agg = src.groupby(['cmdb_id','trace_name']).agg(
            anomaly_count = ('value','size'),
            earliest_anomaly_ts = ('dt','min'),
            latest_anomaly_ts = ('dt','max'),
            max_value = ('value','max'),
            p95 = ('p95','first')
        ).reset_index()
        def sev_fn(row):
            p = row['p95']
            m = row['max_value']
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf
            return m / p
        agg['severity'] = agg.apply(sev_fn, axis=1)
        anomalies_trace_summary_df = agg.sort_values('severity', ascending=False).reset_index(drop=True)
    else:
        # Fallback: try to recompute anomalies_trace_summary_df from trace.csv and trace_aggregates (similar to earlier cell)
        if 'trace_df' not in globals():
            trace_df = pd.read_csv("trace.csv")
        trace_df['timestamp'] = trace_df['timestamp'].astype(int)
        trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
        # ensure trace_aggregates exists or compute global p95
        if 'trace_aggregates' not in globals() or trace_aggregates is None or trace_aggregates.empty:
            grp = trace_df.groupby(['cmdb_id','trace_name'])['value']
            trace_aggregates = pd.concat([
                grp.count().rename('count'),
                trace_df.groupby(['cmdb_id','trace_name'])['dt'].min().rename('earliest_ts'),
                trace_df.groupby(['cmdb_id','trace_name'])['dt'].max().rename('latest_ts'),
                grp.quantile(0.5).rename('p50'),
                grp.quantile(0.95).rename('p95'),
                grp.quantile(0.99).rename('p99'),
            ], axis=1).reset_index().sort_values('count', ascending=False)
        p95_df = trace_aggregates[['cmdb_id','trace_name','p95']].copy()
        p95_df['p95'] = pd.to_numeric(p95_df['p95'], errors='coerce')
        start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
        end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))
        trace_window_df = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()
        trace_with_p95 = trace_window_df.merge(p95_df, on=['cmdb_id','trace_name'], how='left')
        trace_with_p95 = trace_with_p95[trace_with_p95['p95'].notna()].copy()
        trace_with_p95['is_anomaly'] = trace_with_p95['value'] >= trace_with_p95['p95']
        anomalies_trace_df = trace_with_p95[trace_with_p95['is_anomaly']].copy()
        # Aggregate
        agg = anomalies_trace_df.groupby(['cmdb_id','trace_name']).agg(
            anomaly_count = ('value','size'),
            earliest_anomaly_ts = ('dt','min'),
            latest_anomaly_ts = ('dt','max'),
            max_value = ('value','max'),
            p95 = ('p95','first')
        ).reset_index()
        def sev_fn(row):
            p = row['p95']
            m = row['max_value']
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf
            return m / p
        agg['severity'] = agg.apply(sev_fn, axis=1)
        anomalies_trace_summary_df = agg.sort_values('severity', ascending=False).reset_index(drop=True)

# Now filter rows whose trace_name mentions any Tomcat01..Tomcat04
tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# Ensure trace_name column exists
if 'trace_name' not in anomalies_trace_summary_df.columns and 'trace' in anomalies_trace_summary_df.columns:
    anomalies_trace_summary_df = anomalies_trace_summary_df.rename(columns={'trace': 'trace_name'})

# Build boolean mask for any tomcat mention
mask_any = False
for t in tomcats:
    mask_any = mask_any | anomalies_trace_summary_df['trace_name'].str.contains(t, na=False)

trace_anoms_tomcat_df = anomalies_trace_summary_df[mask_any].copy()

# For each Tomcat produce summary row
rows = []
for t in tomcats:
    mask = trace_anoms_tomcat_df['trace_name'].str.contains(t, na=False)
    subset = trace_anoms_tomcat_df[mask]
    if subset.empty:
        continue
    count_groups = len(subset)
    earliest = subset['earliest_anomaly_ts'].min()
    latest = subset['latest_anomaly_ts'].max()
    max_sev = subset['severity'].replace([np.inf, -np.inf], np.inf).max()
    # top up to 3 trace_name examples by severity
    top_examples = subset.sort_values('severity', ascending=False)['trace_name'].unique()[:3]
    top_examples_str = "; ".join(top_examples)
    rows.append({
        'tomcat_id': t,
        'count_of_trace_groups_mentioning_it': count_groups,
        'earliest_anomaly_ts': earliest,
        'latest_anomaly_ts': latest,
        'max_severity_observed': max_sev,
        'top_trace_examples': top_examples_str
    })

tomcat_summary_df = pd.DataFrame(rows)

# Sort by count desc then max_severity desc, limit to 20 rows
if not tomcat_summary_df.empty:
    tomcat_summary_df = tomcat_summary_df.sort_values(
        ['count_of_trace_groups_mentioning_it','max_severity_observed'],
        ascending=[False, False]
    ).head(20).reset_index(drop=True)
else:
    tomcat_summary_df = tomcat_summary_df[[
        'tomcat_id','count_of_trace_groups_mentioning_it','earliest_anomaly_ts',
        'latest_anomaly_ts','max_severity_observed','top_trace_examples'
    ]]

# Final output: tomcat_summary_df (compact) and trace_anoms_tomcat_df stored in kernel for follow-up
tomcat_summary_df

```
Out[5]:
```
```python
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'tomcat_summary_df' not in globals():
    summary = "Tomcat summary not found in kernel."
else:
    df = tomcat_summary_df.copy()
    lines = []
    lines.append("Tomcat trace-anomaly summary (window 2021-03-06 06:00–06:30 UTC+8):")
    lines.append(f"- Tomcats with anomalies: {len(df)}")
    for _, r in df.iterrows():
        tid = r['tomcat_id']
        cnt = int(r['count_of_trace_groups_mentioning_it'])
        e = r['earliest_anomaly_ts']
        l = r['latest_anomaly_ts']
        e_s = pd.to_datetime(e).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(e) else 'N/A'
        l_s = pd.to_datetime(l).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(l) else 'N/A'
        maxsev = r['max_severity_observed']
        maxsev_s = "inf" if np.isinf(maxsev) else f"{maxsev:.6g}"
        examples = r['top_trace_examples']
        lines.append(f"- {tid}: {cnt} trace group(s); earliest={e_s}; latest={l_s}; max_severity={maxsev_s}; examples=[{examples}]")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

tomcat_id  count_of_trace_groups_mentioning_it       earliest_anomaly_ts         latest_anomaly_ts  max_severity_observed                                 top_trace_examples
0  Tomcat03                                    7 2021-03-06 06:09:00+08:00 2021-03-06 06:23:00+08:00               2.826529  trace.from_Tomcat03.duration_mean; trace.from_...
1  Tomcat04                                    6 2021-03-06 06:01:00+08:00 2021-03-06 06:26:00+08:00               4.613687  trace.from_Tomcat04.duration_mean; trace.to_To...
2  Tomcat01                                    3 2021-03-06 06:04:00+08:00 2021-03-06 06:18:00+08:00               1.743948  trace.from_Tomcat01.duration_p95; trace.from_T...
3  Tomcat02                                    2 2021-03-06 06:05:00+08:00 2021-03-06 06:27:00+08:00               1.828508  trace.to_Tomcat02.duration_p95; trace.from_Tom...```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is loaded and has dt column
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
# Ensure kpi_name exists
if 'kpi_name' not in metric_df.columns and 'kpi' in metric_df.columns:
    metric_df = metric_df.rename(columns={'kpi': 'kpi_name'})

# Ensure global metric_aggregates with p95 exists (compute if missing) - MUST compute on full series per rule 9
if 'metric_aggregates' not in globals() or metric_aggregates is None or metric_aggregates.empty:
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_aggregates = pd.concat([
        grp.count().rename('count'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].min().rename('earliest_ts'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].max().rename('latest_ts'),
        grp.quantile(0.5).rename('p50'),
        grp.quantile(0.95).rename('p95'),
        grp.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)
metric_aggregates['p95'] = pd.to_numeric(metric_aggregates['p95'], errors='coerce')

# Define tomcats and incident window (inclusive) in UTC+8
tomcats = ['Tomcat03', 'Tomcat04']
start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))

# Filter metric_df to Tomcat03/04 and time window
mask = metric_df['cmdb_id'].isin(tomcats) & (metric_df['dt'] >= start) & (metric_df['dt'] <= end)
metric_tomcat_window = metric_df[mask].copy()

# Join global p95 thresholds from metric_aggregates (match on cmdb_id and kpi_name)
p95_lookup = metric_aggregates[['cmdb_id','kpi_name','p95']].copy()
metric_tomcat_with_p95 = metric_tomcat_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left')

# Keep only rows with p95 available
metric_tomcat_with_p95 = metric_tomcat_with_p95[metric_tomcat_with_p95['p95'].notna()].copy()

# Mark anomalies: value >= p95
metric_tomcat_with_p95['is_anomaly'] = metric_tomcat_with_p95['value'] >= metric_tomcat_with_p95['p95']

# Store full anomalous points for Tomcats for follow-up
tomcat_metrics_anoms_df = metric_tomcat_with_p95[metric_tomcat_with_p95['is_anomaly']].copy()

# Build compact summary per (cmdb_id, kpi_name)
if not tomcat_metrics_anoms_df.empty:
    summary = tomcat_metrics_anoms_df.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('dt','min'),
        latest_anomaly_ts = ('dt','max'),
        max_value = ('value','max'),
        p95 = ('p95','first')
    ).reset_index()
    # severity = max_value / p95 (handle p95 == 0)
    def sev(row):
        p = row['p95']
        m = row['max_value']
        if pd.isna(p):
            return np.nan
        if p == 0:
            return np.inf
        return m / p
    summary['severity'] = summary.apply(sev, axis=1)
    summary_table = summary.sort_values('severity', ascending=False).head(20).reset_index(drop=True)
else:
    summary_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','p95','severity'
    ])

# Final output: compact summary_table (top 20). tomcat_metrics_anoms_df stored in kernel.
summary_table

```
Out[6]:
```
```python
# Plain-English summary of the metric check for Tomcat03 and Tomcat04
# Uses in-kernel variables: summary_table, tomcat_metrics_anoms_df (recomputed above)
import pandas as pd

if 'summary_table' in globals() and not summary_table.empty:
    msg = ("Metric anomalies detected for Tomcat03/04 in 2021-03-06 06:00–06:30 (UTC+8). "
           "See summary_table for details.")
else:
    # Check the stored anomalies dataframe as additional confirmation
    empty_ok = True
    if 'tomcat_metrics_anoms_df' in globals():
        empty_ok = tomcat_metrics_anoms_df.empty
    if empty_ok:
        msg = ("No metric anomalies found for Tomcat03 or Tomcat04 during 2021-03-06 06:00–06:30 (UTC+8) "
               "when using the global p95 thresholds (value >= p95). "
               "The stored tomcat_metrics_anoms_df is empty.")
    else:
        msg = ("No summary rows, but tomcat_metrics_anoms_df contains anomalous points; "
               "please inspect tomcat_metrics_anoms_df for raw anomalous metric points.")

msg
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, anomaly_count, earliest_anomaly_ts, latest_anomaly_ts, max_value, p95, severity]
Index: []```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

components = ['apache02', 'Redis01', 'Mysql02', 'MG01']
# Incident window inclusive
start = tz.localize(pd.Timestamp('2021-03-06 06:00:00'))
end   = tz.localize(pd.Timestamp('2021-03-06 06:30:00'))

# Ensure metric_df loaded
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'kpi_name' not in metric_df.columns and 'kpi' in metric_df.columns:
    metric_df = metric_df.rename(columns={'kpi': 'kpi_name'})

# Ensure metric_aggregates with global p95 exists (compute on full series if missing)
if 'metric_aggregates' not in globals() or metric_aggregates is None or metric_aggregates.empty:
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_aggregates = pd.concat([
        grp.count().rename('count'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].min().rename('earliest_ts'),
        metric_df.groupby(['cmdb_id','kpi_name'])['dt'].max().rename('latest_ts'),
        grp.quantile(0.5).rename('p50'),
        grp.quantile(0.95).rename('p95'),
        grp.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)
metric_aggregates['p95'] = pd.to_numeric(metric_aggregates['p95'], errors='coerce')

# Ensure trace_df loaded
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'trace_name' not in trace_df.columns and 'kpi_name' in trace_df.columns:
    trace_df = trace_df.rename(columns={'kpi_name': 'trace_name'})

# Ensure trace_aggregates exists (global p95)
if 'trace_aggregates' not in globals() or trace_aggregates is None or trace_aggregates.empty:
    grp_t = trace_df.groupby(['cmdb_id','trace_name'])['value']
    trace_aggregates = pd.concat([
        grp_t.count().rename('count'),
        trace_df.groupby(['cmdb_id','trace_name'])['dt'].min().rename('earliest_ts'),
        trace_df.groupby(['cmdb_id','trace_name'])['dt'].max().rename('latest_ts'),
        grp_t.quantile(0.5).rename('p50'),
        grp_t.quantile(0.95).rename('p95'),
        grp_t.quantile(0.99).rename('p99'),
    ], axis=1).reset_index().sort_values('count', ascending=False)
trace_aggregates['p95'] = pd.to_numeric(trace_aggregates['p95'], errors='coerce')

# Recompute metric anomalies for window and components if anomalies_metric_df missing
if 'anomalies_metric_df' not in globals() or anomalies_metric_df is None:
    # filter metric_df to window and components
    m_window = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end) & (metric_df['cmdb_id'].isin(components))].copy()
    p95_lookup = metric_aggregates[['cmdb_id','kpi_name','p95']].copy()
    m_with_p95 = m_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='left')
    m_with_p95 = m_with_p95[m_with_p95['p95'].notna()].copy()
    m_with_p95['is_anomaly'] = m_with_p95['value'] >= m_with_p95['p95']
    anomalies_metric_df = m_with_p95[m_with_p95['is_anomaly']].copy()
else:
    # ensure anomalies_metric_df is filtered to window and components
    anomalies_metric_df = anomalies_metric_df[
        (anomalies_metric_df['dt'] >= start) & (anomalies_metric_df['dt'] <= end) & (anomalies_metric_df['cmdb_id'].isin(components))
    ].copy()

# Recompute trace anomalies for window and components if anomalies_trace_df missing
if 'anomalies_trace_df' not in globals() or anomalies_trace_df is None:
    t_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end) & (trace_df['cmdb_id'].isin(components))].copy()
    p95_t_lookup = trace_aggregates[['cmdb_id','trace_name','p95']].copy()
    t_with_p95 = t_window.merge(p95_t_lookup, on=['cmdb_id','trace_name'], how='left')
    t_with_p95 = t_with_p95[t_with_p95['p95'].notna()].copy()
    t_with_p95['is_anomaly'] = t_with_p95['value'] >= t_with_p95['p95']
    anomalies_trace_df = t_with_p95[t_with_p95['is_anomaly']].copy()
else:
    anomalies_trace_df = anomalies_trace_df[
        (anomalies_trace_df['dt'] >= start) & (anomalies_trace_df['dt'] <= end) & (anomalies_trace_df['cmdb_id'].isin(components))
    ].copy()

# Store full filtered anomaly points for follow-up as required
candidate_metrics_anoms_df = anomalies_metric_df.copy()
candidate_traces_anoms_df = anomalies_trace_df.copy()

# Helper to compute per-component metric summary
metric_summary_rows = []
for comp in components:
    sub = candidate_metrics_anoms_df[candidate_metrics_anoms_df['cmdb_id'] == comp]
    if sub.empty:
        earliest_metric = pd.NaT
        metric_kpi = None
        max_metric_sev = np.nan
    else:
        earliest_metric = sub['dt'].min()
        # compute severity per kpi_name: severity = max_value / p95
        grp = sub.groupby('kpi_name').agg(max_value=('value','max'), p95=('p95','first')).reset_index()
        def sev_fn(r):
            p = r['p95']
            m = r['max_value']
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf
            return m / p
        grp['severity'] = grp.apply(sev_fn, axis=1)
        # pick kpi with max severity (tie-breaker: higher max_value)
        grp = grp.sort_values(['severity','max_value'], ascending=[False, False]).reset_index(drop=True)
        metric_kpi = grp.loc[0,'kpi_name']
        max_metric_sev = grp.loc[0,'severity']
        max_metric_maxv = grp.loc[0,'max_value']
        max_metric_p95 = grp.loc[0,'p95']
    metric_summary_rows.append({
        'component': comp,
        'earliest_metric_anom_ts': earliest_metric,
        'metric_kpi_with_max_sev': metric_kpi,
        'max_metric_severity': max_metric_sev
    })

metric_summary_df = pd.DataFrame(metric_summary_rows)

# Helper to compute per-component trace summary
trace_summary_rows = []
for comp in components:
    sub = candidate_traces_anoms_df[candidate_traces_anoms_df['cmdb_id'] == comp]
    if sub.empty:
        earliest_trace = pd.NaT
        trace_name = None
        max_trace_sev = np.nan
    else:
        earliest_trace = sub['dt'].min()
        grp = sub.groupby('trace_name').agg(max_value=('value','max'), p95=('p95','first')).reset_index()
        def sev_fn2(r):
            p = r['p95']
            m = r['max_value']
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf
            return m / p
        grp['severity'] = grp.apply(sev_fn2, axis=1)
        grp = grp.sort_values(['severity','max_value'], ascending=[False, False]).reset_index(drop=True)
        trace_name = grp.loc[0,'trace_name']
        max_trace_sev = grp.loc[0,'severity']
    trace_summary_rows.append({
        'component': comp,
        'earliest_trace_anom_ts': earliest_trace,
        'trace_name_with_max_sev': trace_name,
        'max_trace_severity': max_trace_sev
    })

trace_summary_df = pd.DataFrame(trace_summary_rows)

# Merge metric and trace summaries
summary_df = metric_summary_df.merge(trace_summary_df, on='component', how='outer')

# Compute overall earliest anomaly (min of metric and trace earliest) for sorting
def min_dt(a, b):
    if pd.isna(a) and pd.isna(b):
        return pd.NaT
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return min(a, b)

summary_df['earliest_overall_anom_ts'] = summary_df.apply(lambda r: min_dt(r['earliest_metric_anom_ts'], r['earliest_trace_anom_ts']), axis=1)

# Sort by earliest_overall_anom_ts (NaT last)
summary_df = summary_df.sort_values(by=['earliest_overall_anom_ts'], na_position='last').reset_index(drop=True)

# Keep requested columns and limit to these 4 components (up to 4 rows)
out_df = summary_df[[
    'component',
    'earliest_metric_anom_ts',
    'metric_kpi_with_max_sev',
    'max_metric_severity',
    'earliest_trace_anom_ts',
    'trace_name_with_max_sev',
    'max_trace_severity'
]].head(20).reset_index(drop=True)

# Final output variables in kernel: candidate_metrics_anoms_df, candidate_traces_anoms_df, out_df
out_df, candidate_metrics_anoms_df.shape, candidate_traces_anoms_df.shape

```
Out[7]:
```
```python
# Summarize the correlation results for components apache02, Redis01, Mysql02, MG01
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing in-kernel variables: out_df, candidate_metrics_anoms_df, candidate_traces_anoms_df
if 'out_df' not in globals():
    summary = "No correlation summary (out_df) found in kernel."
else:
    df = out_df.copy()
    metric_count = candidate_metrics_anoms_df.shape[0] if 'candidate_metrics_anoms_df' in globals() else 0
    trace_count  = candidate_traces_anoms_df.shape[0] if 'candidate_traces_anoms_df' in globals() else 0

    lines = []
    lines.append("Correlation summary for components (window 2021-03-06 06:00–06:30 UTC+8):")
    lines.append(f"- Source anomaly points used: metric points={metric_count}, trace points={trace_count}.")
    lines.append("")

    for _, r in df.iterrows():
        comp = r['component']
        # metric info
        em = r['earliest_metric_anom_ts']
        if pd.isna(em):
            em_s = "N/A"
        else:
            em_s = pd.to_datetime(em).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        mk = r['metric_kpi_with_max_sev'] if pd.notna(r['metric_kpi_with_max_sev']) else "N/A"
        msev = r['max_metric_severity']
        msev_s = ("inf" if np.isinf(msev) else f"{msev:.6g}") if pd.notna(msev) else "N/A"

        # trace info
        et = r['earliest_trace_anom_ts']
        if pd.isna(et):
            et_s = "N/A"
        else:
            et_s = pd.to_datetime(et).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        tk = r['trace_name_with_max_sev'] if pd.notna(r['trace_name_with_max_sev']) else "N/A"
        tsev = r['max_trace_severity']
        tsev_s = ("inf" if np.isinf(tsev) else f"{tsev:.6g}") if pd.notna(tsev) else "N/A"

        lines.append(f"- {comp}: earliest_metric_anom={em_s}; top_metric={mk} (severity={msev_s}); earliest_trace_anom={et_s}; top_trace={tk} (severity={tsev_s})")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  component   earliest_metric_anom_ts                         metric_kpi_with_max_sev  max_metric_severity    earliest_trace_anom_ts            trace_name_with_max_sev  max_trace_severity
0   Mysql02 2021-03-06 06:00:00+08:00                     OSLinux-CPU_CPU_CPUUserTime             1.062010                       NaT                               None                 NaN
1   Redis01 2021-03-06 06:00:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                  inf                       NaT                               None                 NaN
2  apache02 2021-03-06 06:00:00+08:00                          OSLinux-CPU_CPU_CPUWio             1.973342                       NaT                               None                 NaN
3      MG01                       NaT                                            None                  NaN 2021-03-06 06:01:00+08:00  trace.from_Tomcat04.duration_mean            4.613687, (502, 7), (35, 7))```
```